In [ ]:
%pip install implicit

## ALS Recommender Implementation and Evaluation

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import scipy.sparse as sp
import os
from implicit.als import AlternatingLeastSquares

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import scipy.sparse as sp
import os
from implicit.als import AlternatingLeastSquares

def assign_users_to_groups(user_ids, random_seed=42):
    """
    Randomly assign users to control and treatment groups.

    Args:
        user_ids: Array of user IDs
        random_seed: Random seed for reproducibility

    Returns:
        DataFrame with user_id and group assignment (0=control, 1=treatment)
    """
    np.random.seed(random_seed)
    n_users = len(user_ids)
    assignments = np.random.binomial(n=1, p=0.5, size=n_users)

    return pd.DataFrame(
        {
            'user_id': user_ids,
            'group': assignments
        }
    )


def calculate_hits_at_10(recommendations, actual_reorders):
    """
    Calculate Hits@10: number of recommended items that were actually reordered.

    Args:
        recommendations: List of recommended product IDs (top 10)
        actual_reorders: List of product IDs actually reordered by user

    Returns:
        Number of hits
    """
    rec_set = set(recommendations)
    actual_set = set(actual_reorders)
    return len(rec_set & actual_set)


def calculate_revenue_from_hits(recommendations, actual_reorders):
    """
    Calculate revenue from hits: count of recommended items that were reordered.

    Args:
        recommendations: List of recommended product IDs
        actual_reorders: List of product IDs actually reordered by user

    Returns:
        Revenue metric (count of hits)
    """
    return calculate_hits_at_10(recommendations, actual_reorders)


def calculate_basket_coverage(recommendations, actual_reorders):
    """
    Calculate basket coverage: proportion of actual reorders covered by recommendations.

    Args:
        recommendations: List of recommended product IDs
        actual_reorders: List of product IDs actually reordered by user

    Returns:
        Proportion of basket covered (0-1)
    """
    if len(actual_reorders) == 0:
        return 0.0

    rec_set = set(recommendations)
    actual_set = set(actual_reorders)
    hits = len(rec_set & actual_set)

    return hits / len(actual_set)


def paired_wilcoxon_test(metric_baseline, metric_als, alternative='two-sided'):
    """
    Run paired Wilcoxon signed-rank test comparing metrics between two recommenders.

    Args:
        metric_baseline: Array of metrics from baseline recommender
        metric_als: Array of metrics from ALS recommender
        alternative: Type of test ('two-sided', 'greater', 'less')

    Returns:
        Dictionary with test statistic, p-value, and effect sizes
    """
    metric_baseline = np.array(metric_baseline)
    metric_als = np.array(metric_als)

    # Remove pairs with both values zero
    valid_mask = (metric_baseline != 0) | (metric_als != 0)
    metric_baseline = metric_baseline[valid_mask]
    metric_als = metric_als[valid_mask]

    if len(metric_baseline) == 0:
        return {
            'test': 'Wilcoxon Signed-Rank',
            'statistic': np.nan,
            'p_value': np.nan,
            'baseline_mean': 0,
            'als_mean': 0,
            'baseline_median': 0,
            'als_median': 0,
            'valid_pairs': 0
        }

    statistic, p_value = stats.wilcoxon(metric_baseline, metric_als, alternative=alternative)

    return {
        'test': 'Wilcoxon Signed-Rank',
        'statistic': statistic,
        'p_value': p_value,
        'baseline_mean': metric_baseline.mean(),
        'als_mean': metric_als.mean(),
        'baseline_median': np.median(metric_baseline),
        'als_median': np.median(metric_als),
        'baseline_std': metric_baseline.std(),
        'als_std': metric_als.std(),
        'valid_pairs': len(metric_baseline)
    }


def run_paired_comparison_test(user_ids, baseline_recs_dict, als_recs_dict, ground_truth_csr, user_index, product_index):
    """
    Run paired statistical test comparing baseline vs ALS recommendations.

    Args:
        user_ids: Array of user IDs
        baseline_recs_dict: Dict mapping user_id to baseline recommendations
        als_recs_dict: Dict mapping user_id to ALS recommendations
        ground_truth_csr: Sparse matrix of actual reorders for evaluation (e.g., test set)
        user_index: Array mapping user index to user ID
        product_index: Array mapping internal product index to product ID

    Returns:
        Dictionary with metrics and statistical test results
    """
    # Create mapping from user_id to index in ground_truth_csr
    user_id_to_idx = {uid: idx for idx, uid in enumerate(user_index)}

    # Calculate metrics for each user
    baseline_hits = []
    baseline_revenue = []
    baseline_coverage = []

    als_hits = []
    als_revenue = []
    als_coverage = []

    for user_id in user_ids:
        # Get user's actual reorders from the ground_truth_csr
        user_idx = user_id_to_idx.get(user_id)
        if user_idx is None:
            continue

        # Convert internal product indices from ground_truth_csr to actual product IDs
        actual_reorder_internal_indices = ground_truth_csr[user_idx].indices
        actual_reorders = product_index[actual_reorder_internal_indices].tolist()

        # Baseline metrics
        baseline_recs = baseline_recs_dict.get(user_id, [])
        baseline_hits.append(calculate_hits_at_10(baseline_recs, actual_reorders))
        baseline_revenue.append(calculate_revenue_from_hits(baseline_recs, actual_reorders))
        baseline_coverage.append(calculate_basket_coverage(baseline_recs, actual_reorders))

        # ALS metrics
        als_recs = als_recs_dict.get(user_id, [])
        als_hits.append(calculate_hits_at_10(als_recs, actual_reorders))
        als_revenue.append(calculate_revenue_from_hits(als_recs, actual_reorders))
        als_coverage.append(calculate_basket_coverage(als_recs, actual_reorders))

    # Run paired tests for each metric
    hits_test = paired_wilcoxon_test(baseline_hits, als_hits)
    revenue_test = paired_wilcoxon_test(baseline_revenue, als_revenue)
    coverage_test = paired_wilcoxon_test(baseline_coverage, als_coverage)

    return {
        'baseline_hits': baseline_hits,
        'als_hits': als_hits,
        'hits_test': hits_test,
        'baseline_revenue': baseline_revenue,
        'als_revenue': als_revenue,
        'revenue_test': revenue_test,
        'baseline_coverage': baseline_coverage,
        'als_coverage': als_coverage,
        'coverage_test': coverage_test
    }


def load_reorder_matrix_and_indices():
    """Load the saved CSR matrix and indices from data/processed folder."""
    csr_path = "/content/reorder_csr.npz"
    user_index_path = "/content/reorder_user_index.npy"
    product_index_path = "/content/reorder_product_index.npy"

    reorder_csr = sp.load_npz(csr_path).astype(np.float32)
    user_index = np.load(user_index_path)
    product_index = np.load(product_index_path)

    return reorder_csr, user_index, product_index

def create_temporal_split_csr(reorder_csr, test_split_ratio=0.2, random_seed=42):
    """
    Splits the reorder_csr into training and testing CSR matrices using a random split.
    For each user, a percentage of their interactions are randomly assigned to the test set.

    Args:
        reorder_csr: The original sparse matrix of user-item interactions.
        test_split_ratio: The proportion of interactions to put into the test set (0.0 to 1.0).
        random_seed: Random seed for reproducibility.

    Returns:
        train_csr, test_csr: Sparse matrices for training and testing.
    """
    np.random.seed(random_seed)
    num_users, num_items = reorder_csr.shape
    train_rows, train_cols, train_data = [], [], []
    test_rows, test_cols, test_data = [], [], []

    for user_idx in range(num_users):
        user_interactions = reorder_csr.getrow(user_idx).indices
        num_user_interactions = len(user_interactions)

        if num_user_interactions < 2:
            # Users with 0 or 1 interaction are entirely assigned to the training set.
            # They cannot form a meaningful test set with at least one item.
            for item_idx in user_interactions:
                train_rows.append(user_idx)
                train_cols.append(item_idx)
                train_data.append(1) # Assuming binary interactions
            continue # Skip to next user

        # For users with 2 or more interactions, proceed with splitting
        np.random.shuffle(user_interactions)

        # Ensure at least one item in training and at least one item in test
        train_count = max(1, int(num_user_interactions * (1 - test_split_ratio)))
        # If forcing train_count to 1 makes test_count 0, adjust train_count down by 1
        if num_user_interactions - train_count == 0:
            train_count = num_user_interactions - 1
        test_count = num_user_interactions - train_count

        train_interactions = user_interactions[:train_count]
        test_interactions = user_interactions[train_count:]

        # Add to train_csr
        for item_idx in train_interactions:
            train_rows.append(user_idx)
            train_cols.append(item_idx)
            train_data.append(1) # Assuming binary interactions

        # Add to test_csr
        for item_idx in test_interactions:
            test_rows.append(user_idx)
            test_cols.append(item_idx)
            test_data.append(1) # Assuming binary interactions

    train_csr = sp.csr_matrix((train_data, (train_rows, train_cols)), shape=(num_users, num_items), dtype=np.float32)
    test_csr = sp.csr_matrix((test_data, (test_rows, test_cols)), shape=(num_users, num_items), dtype=np.float32)

    return train_csr, test_csr


def get_popularity_based_recommendations(train_csr, product_index, n_recs=10):
    """
    Generate baseline recommendations based on product popularity from the training set.
    """
    # Calculate product popularity (sum of reorders across all users) based on the training data
    product_popularity = np.asarray(train_csr.sum(axis=0)).flatten()

    # Sort products by popularity
    top_products_idx = np.argsort(product_popularity)[::-1][:n_recs]
    top_products = product_index[top_products_idx]

    return top_products.tolist()


def generate_baseline_recommendations(train_csr, product_index, user_index, n_recs=10):
    """
    Generate baseline (popularity-based) recommendations for all users based on training data.
    """
    baseline_recs = {}
    top_products = get_popularity_based_recommendations(train_csr, product_index, n_recs)

    for user_id in user_index:
        baseline_recs[user_id] = top_products

    return baseline_recs


def train_als_model(train_csr):
    """
    Train ALS model on the training reorder matrix.
    """
    alpha = 40
    # The implicit library expects a user-item matrix where rows are users and columns are items.
    # train_csr is already users x products. Create confidence matrix from it.
    confidence_matrix = train_csr.copy()
    confidence_matrix.data = 1 + alpha * confidence_matrix.data

    als = AlternatingLeastSquares(
        factors=64,
        regularization=0.05,
        iterations=30
    )
    # Train directly on the user-item confidence matrix
    # Removed: item_user_matrix = confidence_matrix.T
    als.fit(confidence_matrix) # Pass the confidence_matrix directly, which is users x items

    return als, confidence_matrix


def generate_als_recommendations(train_csr, als_model, user_index, product_index, n_recs=10):
    als_recs = {}

    for user_idx, user_id in enumerate(user_index):
        # Get the items the user has already interacted with from the TRAIN CSR matrix
        # These are internal product indices for the current user_idx
        user_items_internal_indices = train_csr[user_idx]

        # Use the implicit.recommend method, which automatically filters already-liked items
        recommended_internal_indices, _ = als_model.recommend(
            userid=user_idx,
            user_items=user_items_internal_indices, # Pass the sparse row for the user from TRAIN_CSR
            N=n_recs,
            filter_already_liked_items=True # Explicitly ensure already-liked items are filtered
        )

        # Convert internal item indices back to original product IDs
        recommended_product_ids = product_index[recommended_internal_indices].tolist()
        als_recs[user_id] = recommended_product_ids

    return als_recs

In [ ]:
print("Loading reorder matrix and indices...")
reorder_csr, user_index, product_index = load_reorder_matrix_and_indices()

print(f"Loaded full matrix shape: {reorder_csr.shape}")
print(f"Number of users: {len(user_index)}")
print(f"Number of products: {len(product_index)}")

print("\nSplitting data into training and test sets...")
train_csr, test_csr = create_temporal_split_csr(reorder_csr, test_split_ratio=0.2)
print(f"Train matrix shape: {train_csr.shape}")
print(f"Test matrix shape: {test_csr.shape}")

# Calculate and print average actual reorders per user in the test set
num_users_in_test_set = test_csr.getnnz(axis=1).astype(bool).sum()
if num_users_in_test_set > 0:
    avg_test_reorders = test_csr.sum() / num_users_in_test_set
    print(f"Average actual reorders per user in test set (for evaluated users): {avg_test_reorders:.2f}")
else:
    print("No users in the test set with actual reorders.")

print("\n" + "="*80)
print("GENERATING RECOMMENDATIONS")
print("="*80)

print("\nGenerating popularity-based recommendations for all users (based on training data)...")
baseline_recs_dict = generate_baseline_recommendations(
    train_csr, product_index, user_index, n_recs=10
)
print(f"✓ Generated baseline recommendations for {len(baseline_recs_dict)} users")

print("\nTraining ALS model on training data...")
als_model, confidence_matrix = train_als_model(train_csr)

print("Generating ALS recommendations for all users (filtering items seen in training data)...")
als_recs_dict = generate_als_recommendations(
    train_csr, als_model, user_index, product_index, n_recs=10
)
print(f"✓ Generated ALS recommendations for {len(als_recs_dict)} users")

print("\n" + "="*80)
print("PAIRED STATISTICAL TEST")
print("="*80)

print("\nCalculating metrics and running paired Wilcoxon signed-rank tests (evaluating against test set)...")
results = run_paired_comparison_test(user_index, baseline_recs_dict, als_recs_dict,
                                    test_csr, user_index, product_index)

print("\n" + "-"*80)
print("HITS@10 TEST")
print("-"*80)
print(f"Baseline (Popularity):")
print(f"  Mean: {results['hits_test']['baseline_mean']:.4f}")
print(f"  Median: {results['hits_test']['baseline_median']:.4f}")
print(f"  Std Dev: {results['hits_test']['baseline_std']:.4f}")

print(f"\nALS Recommender:")
print(f"  Mean: {results['hits_test']['als_mean']:.4f}")
print(f"  Median: {results['hits_test']['als_median']:.4f}")
print(f"  Std Dev: {results['hits_test']['als_std']:.4f}")

print(f"\nPaired Wilcoxon Test (n={results['hits_test']['valid_pairs']} pairs):")
print(f"  Test Statistic: {results['hits_test']['statistic']:.2f}")
print(f"  P-value: {results['hits_test']['p_value']:.6f}")
if results['hits_test']['p_value'] < 0.05:
    print(f"  ✓ SIGNIFICANT (p < 0.05)")
    if results['hits_test']['als_mean'] > results['hits_test']['baseline_mean']:
        print(f"    ALS performs BETTER than Popularity")
    else:
        print(f"    Popularity performs BETTER than ALS")
else:
    print(f"  ✗ NOT SIGNIFICANT (p \u2265 0.05)")

print("\n" + "-"*80)
print("REVENUE FROM HITS TEST")
print("-"*80)
print(f"Baseline (Popularity):")
print(f"  Mean: {results['revenue_test']['baseline_mean']:.4f}")
print(f"  Median: {results['revenue_test']['baseline_median']:.4f}")
print(f"  Std Dev: {results['revenue_test']['baseline_std']:.4f}")

print(f"\nALS Recommender:")
print(f"  Mean: {results['revenue_test']['als_mean']:.4f}")
print(f"  Median: {results['revenue_test']['als_median']:.4f}")
print(f"  Std Dev: {results['revenue_test']['als_std']:.4f}")

print(f"\nPaired Wilcoxon Test (n={results['revenue_test']['valid_pairs']} pairs):")
print(f"  Test Statistic: {results['revenue_test']['statistic']:.2f}")
print(f"  P-value: {results['revenue_test']['p_value']:.6f}")
if results['revenue_test']['p_value'] < 0.05:
    print(f"  ✓ SIGNIFICANT (p < 0.05)")
    if results['revenue_test']['als_mean'] > results['revenue_test']['baseline_mean']:
        print(f"    ALS generates MORE revenue from hits")
    else:
        print(f"    Popularity generates MORE revenue from hits")
else:
    print(f"  ✗ NOT SIGNIFICANT (p \u2265 0.05)")

print("\n" + "-"*80)
print("BASKET COVERAGE TEST")
print("-"*80)
print(f"Baseline (Popularity):")
print(f"  Mean: {results['coverage_test']['baseline_mean']:.4f}")
print(f"  Median: {results['coverage_test']['baseline_median']:.4f}")
print(f"  Std Dev: {results['coverage_test']['baseline_std']:.4f}")

print(f"\nALS Recommender:")
print(f"  Mean: {results['coverage_test']['als_mean']:.4f}")
print(f"  Median: {results['coverage_test']['als_median']:.4f}")
print(f"  Std Dev: {results['coverage_test']['als_std']:.4f}")

print(f"\nPaired Wilcoxon Test (n={results['coverage_test']['valid_pairs']} pairs):")
print(f"  Test Statistic: {results['coverage_test']['statistic']:.2f}")
print(f"  P-value: {results['coverage_test']['p_value']:.6f}")
if results['coverage_test']['p_value'] < 0.05:
    print(f"  ✓ SIGNIFICANT (p < 0.05)")
    if results['coverage_test']['als_mean'] > results['coverage_test']['baseline_mean']:
        print(f"    ALS provides BETTER basket coverage")
    else:
        print(f"    Popularity provides BETTER basket coverage")
else:
    print(f"  ✗ NOT SIGNIFICANT (p \u2265 0.05)")

print("\n" + "="*80)

Loading reorder matrix and indices...
Loaded full matrix shape: (203026, 25620)
Number of users: 203026
Number of products: 25620

Splitting data into training and test sets...
Train matrix shape: (203026, 25620)
Test matrix shape: (203026, 25620)
Average actual reorders per user in test set (for evaluated users): 5.76

GENERATING RECOMMENDATIONS

Generating popularity-based recommendations for all users (based on training data)...
✓ Generated baseline recommendations for 203026 users

Training ALS model on training data...


  0%|          | 0/30 [00:00<?, ?it/s]

Generating ALS recommendations for all users (filtering items seen in training data)...
✓ Generated ALS recommendations for 203026 users

PAIRED STATISTICAL TEST

Calculating metrics and running paired Wilcoxon signed-rank tests (evaluating against test set)...

--------------------------------------------------------------------------------
HITS@10 TEST
--------------------------------------------------------------------------------
Baseline (Popularity):
  Mean: 0.7783
  Median: 1.0000
  Std Dev: 0.7399

ALS Recommender:
  Mean: 1.0252
  Median: 1.0000
  Std Dev: 0.7976

Paired Wilcoxon Test (n=91436 pairs):
  Test Statistic: 898461567.00
  P-value: 0.000000
  ✓ SIGNIFICANT (p < 0.05)
    ALS performs BETTER than Popularity

--------------------------------------------------------------------------------
REVENUE FROM HITS TEST
--------------------------------------------------------------------------------
Baseline (Popularity):
  Mean: 0.7783
  Median: 1.0000
  Std Dev: 0.7399

ALS 

In [ ]:
print("Debugging generate_als_recommendations for potential IndexError...")

# Verify product_index size
max_valid_product_index = len(product_index) - 1
print(f"Max valid index for product_index: {max_valid_product_index} (total products: {len(product_index)})")

problem_found = False
# Check a sample of users to quickly identify if the issue is systematic.
# We'll pick 100 random users or all users if there are fewer than 100.
sample_size = min(100, len(user_index))
sample_user_indices_for_debug = np.random.choice(len(user_index), size=sample_size, replace=False)

for user_idx_debug in sample_user_indices_for_debug:
    user_id_debug = user_index[user_idx_debug]
    user_items_internal_indices_debug = reorder_csr[user_idx_debug]

    try:
        # Call als_model.recommend
        recommended_internal_indices_debug, _ = als_model.recommend(
            userid=user_idx_debug,
            user_items=user_items_internal_indices_debug,
            N=10, # Using 10 as per the original call
            filter_already_liked_items=True
        )

        if len(recommended_internal_indices_debug) > 0:
            max_rec_idx = np.max(recommended_internal_indices_debug)
            min_rec_idx = np.min(recommended_internal_indices_debug)

            # Check for out-of-bounds indices
            if max_rec_idx > max_valid_product_index or min_rec_idx < 0:
                print(f"\nERROR: Out-of-bounds internal index found for user_idx {user_idx_debug} (user_id {user_id_debug})")
                print(f"  Recommended internal indices: {recommended_internal_indices_debug}")
                print(f"  Max recommended index: {max_rec_idx}, Min recommended index: {min_rec_idx}")
                problem_found = True
                break # Found the problem, stop and report

        # Attempt the product_index lookup to catch any subtle issues
        _ = product_index[recommended_internal_indices_debug].tolist()

    except IndexError as e:
        print(f"\nIndexError encountered for user_idx {user_idx_debug} (user_id {user_id_debug}) during product_index lookup: {e}")
        if 'recommended_internal_indices_debug' in locals():
            print(f"  Recommended internal indices: {recommended_internal_indices_debug}")
        problem_found = True
        break # Found the problem, stop and report
    except Exception as e:
        print(f"\nUnexpected error for user_idx {user_idx_debug} (user_id {user_id_debug}): {type(e).__name__} - {e}")
        problem_found = True
        break

if not problem_found:
    print(f"\nNo obvious IndexError found for the {sample_size} sampled users in generate_als_recommendations.")
    print("The issue might be very specific to certain users not covered by the sample, or it might be occurring in the 'run_paired_comparison_test' function itself.")

# You can extend this to check all users if needed, but it might take longer:
# if not problem_found:
#     print("Checking all users for potential IndexError (this may take a while)...")
#     # You would repeat the loop above for all user_idx in range(len(user_index))

Debugging generate_als_recommendations for potential IndexError...
Max valid index for product_index: 25619 (total products: 25620)

No obvious IndexError found for the 100 sampled users in generate_als_recommendations.
The issue might be very specific to certain users not covered by the sample, or it might be occurring in the 'run_paired_comparison_test' function itself.


## SASRec (Self-Attentive Sequential Recommendation)

SASRec is a sequential recommendation model that leverages the self-attention mechanism (from the Transformer architecture) to capture the sequential dependencies of user-item interactions. Unlike traditional collaborative filtering methods, SASRec models a user's next item choice based on their *past sequence* of interactions, making it particularly effective for capturing short-term, dynamic preferences.

Key characteristics:
- **Self-Attention:** Allows the model to weigh the importance of different past items when predicting the next item, capturing long-range dependencies in a sequence.
- **Positional Encoding:** Incorporates information about the order of items in the sequence.
- **Item Embeddings:** Each item is represented by a dense vector (embedding), which the model learns during training.

Next, we'll define the SASRec model architecture using TensorFlow/Keras.

In [ ]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1) # Shape: (max_len, 1, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x): # x shape: (seq_len, batch_size, d_model)
        # This forward method is not directly used by SASRecPyTorch as designed earlier,
        # where position_embeddings.pe is accessed directly for broadcasting.
        # However, keeping it consistent with how it was defined previously.
        return x

class SASRecPyTorch(nn.Module):
    def __init__(self, num_items, max_seq_len, d_model, num_heads, num_blocks, dropout_rate):
        super(SASRecPyTorch, self).__init__()

        self.item_embeddings = nn.Embedding(num_items + 1, d_model, padding_idx=0)
        # Instantiate PositionalEncoding
        self.position_embeddings = PositionalEncoding(d_model, max_len=max_seq_len)
        self.dropout = nn.Dropout(dropout_rate)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=d_model, # Using d_model as feedforward dimension
            dropout=dropout_rate,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_blocks)

        self.final_layer = nn.Linear(d_model, num_items + 1) # Predicts logits for each item

    def forward(self, input_sequence):
        # input_sequence shape: (batch_size, seq_len)

        # Create a mask for padding (0s) in the input sequence
        # (batch_size, seq_len) -> (batch_size, seq_len) bool
        src_key_padding_mask = (input_sequence == 0)

        item_embed = self.item_embeddings(input_sequence) # (batch_size, seq_len, d_model)

        # SASRec paper uses position embedding added to item embedding *before* dropout
        # Get positional embeddings
        seq_len = input_sequence.size(1)
        # self.position_embeddings.pe has shape (max_len, 1, d_model)
        # We need (1, seq_len, d_model) for broadcasting with batch_first=True item_embed
        pos_embed = self.position_embeddings.pe[:seq_len, :].transpose(0, 1) # Shape: (1, seq_len, d_model)

        x = item_embed + pos_embed # (batch_size, seq_len, d_model)
        x = self.dropout(x)

        # PyTorch TransformerEncoderLayer expects (batch_first=True) (batch_size, seq_len, d_model)
        # We set batch_first=True in the encoder_layer constructor
        transformer_output = self.transformer_encoder(
            x,
            src_key_padding_mask=src_key_padding_mask # Mask out padded elements
        ) # (batch_size, seq_len, d_model)

        # Take the output corresponding to the last item in the sequence for prediction
        # The true SASRec predicts the next item after the *last non-padded* item.
        # For simplicity and consistency with the previous TF implementation,
        # we take the last output of the fixed-length sequence.
        last_item_output = transformer_output[:, -1, :]

        logits = self.final_layer(last_item_output)
        return logits

## SASRec Implementation using PyTorch

Here's the PyTorch implementation of the SASRec model, replacing the previous TensorFlow/Keras version. This will require adapting the data preparation and training steps accordingly.

First, let's import the necessary PyTorch libraries.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math

# Check if CUDA is available and set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1) # Shape: (max_len, 1, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x): # x shape: (seq_len, batch_size, d_model)
        return x + self.pe[:x.size(0), :]

In [ ]:
class SASRecPyTorch(nn.Module):
    def __init__(self, num_items, max_seq_len, d_model, num_heads, num_blocks, dropout_rate):
        super(SASRecPyTorch, self).__init__()

        self.item_embeddings = nn.Embedding(num_items + 1, d_model, padding_idx=0)
        self.position_embeddings = PositionalEncoding(d_model, max_len=max_seq_len)
        self.dropout = nn.Dropout(dropout_rate)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=d_model, # Using d_model as feedforward dimension
            dropout=dropout_rate,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_blocks)

        self.final_layer = nn.Linear(d_model, num_items + 1) # Predicts logits for each item

    def forward(self, input_sequence):
        # input_sequence shape: (batch_size, seq_len)

        # Create a mask for padding (0s) in the input sequence
        # (batch_size, seq_len) -> (batch_size, seq_len) bool
        src_key_padding_mask = (input_sequence == 0)

        item_embed = self.item_embeddings(input_sequence) # (batch_size, seq_len, d_model)

        # SASRec paper uses position embedding added to item embedding *before* dropout
        # Get positional embeddings
        seq_len = input_sequence.size(1)
        # self.position_embeddings.pe has shape (max_len, 1, d_model)
        # We need (1, seq_len, d_model) for broadcasting with batch_first=True item_embed
        pos_embed = self.position_embeddings.pe[:seq_len, :].transpose(0, 1) # Shape: (1, seq_len, d_model)

        x = item_embed + pos_embed # (batch_size, seq_len, d_model)
        x = self.dropout(x)

        # PyTorch TransformerEncoderLayer expects (batch_first=True) (batch_size, seq_len, d_model)
        # We set batch_first=True in the encoder_layer constructor
        transformer_output = self.transformer_encoder(
            x,
            src_key_padding_mask=src_key_padding_mask # Mask out padded elements
        ) # (batch_size, seq_len, d_model)

        # Take the output corresponding to the last item in the sequence for prediction
        # The true SASRec predicts the next item after the *last non-padded* item.
        # For simplicity and consistency with the previous TF implementation,
        # we take the last output of the fixed-length sequence.
        last_item_output = transformer_output[:, -1, :]

        logits = self.final_layer(last_item_output)
        return logits

### Adapting Data Preparation for PyTorch

The `create_sequential_data` function previously generated NumPy arrays (`X_train`, `y_train`, etc.). For PyTorch, we'll typically want to create `torch.Tensor` objects and potentially use `torch.utils.data.Dataset` and `torch.utils.data.DataLoader` for efficient batching and handling of data.

The `create_sequential_data` function would need to be modified to output `torch.Tensor`s, and then we'd wrap them in a `Dataset` and `DataLoader`.

### PyTorch Training Loop

Instead of `model.compile()` and `model.fit()` as in Keras, PyTorch models are trained with an explicit training loop. This involves:

1.  **Defining an Optimizer:** e.g., `optimizer = optim.Adam(model.parameters(), lr=0.001)`
2.  **Defining a Loss Function:** e.g., `criterion = nn.CrossEntropyLoss(ignore_index=0)` (to ignore padding item 0)
3.  **Iterating over Epochs:**
    *   For each epoch, iterate over batches from the `DataLoader`.
    *   Move data to the `device` (GPU if available).
    *   Perform a forward pass to get predictions.
    *   Calculate the loss.
    *   Perform a backward pass (`loss.backward()`).
    *   Update model parameters (`optimizer.step()`).
    *   Zero gradients (`optimizer.zero_grad()`).

### Generating Recommendations with PyTorch

The `generate_sasrec_recommendations` function would also need to be adapted to:

1.  Convert input sequences to `torch.Tensor`s and move them to the `device`.
2.  Set the model to evaluation mode (`model.eval()`).
3.  Perform inference to get item scores (logits).
4.  Convert the output scores back to NumPy if needed for evaluation metrics.

Due to the significant refactoring required for data preparation and the training loop, I've provided the PyTorch model definition. The subsequent cells would need to be updated to use this PyTorch model and its corresponding training/inference patterns.

Now that the SASRec model is defined, the next crucial step is to prepare the data in a sequential format that the model can understand. This involves transforming the `reorder_csr` matrix into sequences of item interactions for each user. We'll also need to create a mapping from internal product indices to sequential item IDs and vice-versa. Additionally, we'll split the data into training, validation, and test sets.

In [ ]:
import numpy as np
from torch.utils.data import Dataset, DataLoader

# Define a PyTorch Dataset for sequential data
class SequentialDataset(Dataset):
    def __init__(self, sequences, targets):
        self.sequences = torch.tensor(sequences, dtype=torch.long)
        self.targets = torch.tensor(targets, dtype=torch.long)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.targets[idx]

def create_sequential_data(reorder_csr, user_index, product_index, max_seq_len=50, batch_size=256):
    """
    Creates sequential data for SASRec from the CSR matrix, adapted for PyTorch.

    Args:
        reorder_csr: Sparse matrix of reorders (users x products).
        user_index: Array mapping internal user index to user ID.
        product_index: Array mapping internal product index to product ID.
        max_seq_len: Maximum sequence length for SASRec.
        batch_size: Batch size for PyTorch DataLoaders.

    Returns:
        train_loader, val_loader, test_loader: PyTorch DataLoaders.
        item_to_id: Dictionary mapping original product ID to a new sequential item ID.
        id_to_item: Dictionary mapping sequential item ID back to original product ID.
        num_unique_items: Total number of unique items (for embedding layer).
        user_sequences_map: Dictionary mapping original user ID to their sequence of item IDs (for recommendations).
    """
    user_sequences_map_for_recs = {} # Will store original product IDs for recommendation function
    all_product_ids_in_csr = []

    for user_idx in range(reorder_csr.shape[0]):
        user_id = user_index[user_idx]
        product_indices = reorder_csr[user_idx].indices
        user_product_ids = product_index[product_indices].tolist()
        if user_product_ids:
            user_sequences_map_for_recs[user_id] = user_product_ids
            all_product_ids_in_csr.extend(user_product_ids)

    unique_product_ids = sorted(list(set(all_product_ids_in_csr)))
    item_to_id = {prod_id: i + 1 for i, prod_id in enumerate(unique_product_ids)} # +1 for padding
    id_to_item = {i + 1: prod_id for i, prod_id in enumerate(unique_product_ids)}

    sequences_pytorch = [] # List of (input_sequence_tensor, target_item_tensor)
    for user_id in user_index:
        if user_id in user_sequences_map_for_recs:
            original_seq = user_sequences_map_for_recs[user_id]
            sequential_ids = [item_to_id[pid] for pid in original_seq if pid in item_to_id]

            if len(sequential_ids) > 1:
                for i in range(1, len(sequential_ids)):
                    input_seq = sequential_ids[:i]
                    target_item = sequential_ids[i]

                    if len(input_seq) > max_seq_len:
                        input_seq = input_seq[-max_seq_len:]

                    # Pad the sequence
                    padded_input_seq = np.zeros(max_seq_len, dtype=np.long)
                    padded_input_seq[max_seq_len - len(input_seq):] = input_seq

                    sequences_pytorch.append((padded_input_seq, target_item))

    np.random.shuffle(sequences_pytorch)

    # Split into train, val, test
    train_size = int(0.8 * len(sequences_pytorch))
    val_size = int(0.1 * len(sequences_pytorch))

    train_data = sequences_pytorch[:train_size]
    val_data = sequences_pytorch[train_size:train_size + val_size]
    test_data = sequences_pytorch[train_size + val_size:]

    X_train = np.array([s[0] for s in train_data])
    y_train = np.array([s[1] for s in train_data])
    X_val = np.array([s[0] for s in val_data])
    y_val = np.array([s[1] for s in val_data])
    X_test = np.array([s[0] for s in test_data])
    y_test = np.array([s[1] for s in test_data])

    train_dataset = SequentialDataset(X_train, y_train)
    val_dataset = SequentialDataset(X_val, y_val)
    test_dataset = SequentialDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    print(f"Total unique items for SASRec: {len(unique_product_ids)}")
    print(f"Total sequences created: {len(sequences_pytorch)}")

    return train_loader, val_loader, test_loader, item_to_id, id_to_item, len(unique_product_ids), user_sequences_map_for_recs

def generate_sasrec_recommendations(sasrec_model, user_sequences_map_full_history, user_index, item_to_id, id_to_item, max_seq_len, n_recs=10, train_csr=None, product_index_global=None):
    sasrec_model.eval() # Set model to evaluation mode
    sasrec_recs = {}

    # Pick one user to debug in detail. Ensure this user has some sequence.
    debug_user_id = None
    for uid in user_index:
        if uid in user_sequences_map_full_history and len(user_sequences_map_full_history[uid]) >= 2:
            debug_user_id = uid
            break
    debug_mode = (debug_user_id is not None) # Activate debug if a suitable user is found

    user_id_to_idx_csr = {uid: idx for idx, uid in enumerate(user_index)} # Mapping to CSR internal index

    with torch.no_grad(): # Disable gradient calculation for inference
        for user_id in user_index:
            is_debug_user = (user_id == debug_user_id) and debug_mode

            if user_id in user_sequences_map_full_history:
                original_seq_full_history = user_sequences_map_full_history[user_id]
                sequential_ids_full_history = [item_to_id[pid] for pid in original_seq_full_history if pid in item_to_id]

                if is_debug_user: print(f"\nDebugging for user {user_id}")
                if is_debug_user: print(f"  Original sequence (products, full history): {original_seq_full_history}")
                if is_debug_user: print(f"  Sequential IDs (mapped, full history): {sequential_ids_full_history}")

                if not sequential_ids_full_history or len(sequential_ids_full_history) < 2: # Need at least 2 items to predict next
                    sasrec_recs[user_id] = []
                    if is_debug_user: print("  Skipping: Not enough sequential IDs for recommendation.")
                    continue

                # Use the last part of the full history sequence for prediction input
                input_seq = sequential_ids_full_history[-max_seq_len:]
                if is_debug_user: print(f"  Input sequence for prediction (sequential IDs): {input_seq}")

                # Pad the input sequence to max_seq_len
                padded_input_seq = np.zeros(max_seq_len, dtype=np.long)
                padded_input_seq[max_seq_len - len(input_seq):] = input_seq

                input_tensor = torch.tensor(np.array([padded_input_seq]), dtype=torch.long).to(device)
                if is_debug_user: print(f"  Input tensor shape: {input_tensor.shape}")

                predictions = sasrec_model(input_tensor)
                item_scores = predictions[0].cpu().numpy() # Move to CPU and convert to numpy
                if is_debug_user: print(f"  Raw item scores shape: {item_scores.shape}")
                if is_debug_user: print(f"  Max raw score: {np.max(item_scores):.4f}, Min raw score: {np.min(item_scores):.4f}")

                # --- CRUCIAL CHANGE HERE: seen_product_ids should come from training interactions ---
                seen_product_ids_for_user = set()
                if train_csr is not None and product_index_global is not None:
                    user_csr_idx = user_id_to_idx_csr.get(user_id)
                    if user_csr_idx is not None:
                        train_interactions_internal_indices = train_csr[user_csr_idx].indices
                        seen_product_ids_for_user = set(product_index_global[train_interactions_internal_indices].tolist())
                else: # Fallback if train_csr not provided (not ideal for evaluation)
                     # This original logic was the problem; it filters out all items ever seen.
                     seen_product_ids_for_user = set(original_seq_full_history)

                if is_debug_user: print(f"  Seen product IDs (from train_csr for filtering): {seen_product_ids_for_user}")


                filtered_scores = np.copy(item_scores)

                # Set score of padding item (ID 0) to -inf to ensure it's not recommended
                if filtered_scores.shape[0] > 0:
                    filtered_scores[0] = -np.inf

                # Set scores of seen items (from training data) to a very low value
                for seen_prod_id in seen_product_ids_for_user:
                    if seen_prod_id in item_to_id: # Only filter if the seen item is part of our model's vocabulary
                        seq_id_of_seen_item = item_to_id[seen_prod_id]
                        if 0 < seq_id_of_seen_item < len(filtered_scores):
                            filtered_scores[seq_id_of_seen_item] = -np.inf

                if is_debug_user: print(f"  Max filtered score after seen item filtering: {np.max(filtered_scores[1:]):.4f} (excluding padding)")
                if is_debug_user: print(f"  Number of non-inf scores after filtering seen items: {np.sum(filtered_scores != -np.inf)}")

                # Get top N recommended item IDs (sequential IDs)
                top_n_sequential_ids = np.argsort(filtered_scores)[::-1]
                if is_debug_user: print(f"  Top sequential IDs (raw from arg_sort): {top_n_sequential_ids[:n_recs+5]}")

                recommended_product_ids = []
                for seq_id in top_n_sequential_ids:
                    if seq_id == 0: # Skip padding item ID
                        continue
                    original_prod_id = id_to_item.get(seq_id)
                    if original_prod_id is not None and original_prod_id not in seen_product_ids_for_user: # Ensure not a seen item
                        recommended_product_ids.append(original_prod_id)
                        if len(recommended_product_ids) >= n_recs:
                            break

                if is_debug_user: print(f"  Final recommended product IDs for user {user_id}: {recommended_product_ids}")
                sasrec_recs[user_id] = recommended_product_ids
            else:
                sasrec_recs[user_id] = [] # No interactions for this user

    return sasrec_recs

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import scipy.sparse as sp
import os
from implicit.als import AlternatingLeastSquares

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import scipy.sparse as sp
import os
from implicit.als import AlternatingLeastSquares

def assign_users_to_groups(user_ids, random_seed=42):
    """
    Randomly assign users to control and treatment groups.

    Args:
        user_ids: Array of user IDs
        random_seed: Random seed for reproducibility

    Returns:
        DataFrame with user_id and group assignment (0=control, 1=treatment)
    """
    np.random.seed(random_seed)
    n_users = len(user_ids)
    assignments = np.random.binomial(n=1, p=0.5, size=n_users)

    return pd.DataFrame(
        {
            'user_id': user_ids,
            'group': assignments
        }
    )


def calculate_hits_at_10(recommendations, actual_reorders):
    """
    Calculate Hits@10: number of recommended items that were actually reordered.

    Args:
        recommendations: List of recommended product IDs (top 10)
        actual_reorders: List of product IDs actually reordered by user

    Returns:
        Number of hits
    """
    rec_set = set(recommendations)
    actual_set = set(actual_reorders)
    return len(rec_set & actual_set)


def calculate_revenue_from_hits(recommendations, actual_reorders):
    """
    Calculate revenue from hits: count of recommended items that were reordered.

    Args:
        recommendations: List of recommended product IDs
        actual_reorders: List of product IDs actually reordered by user

    Returns:
        Revenue metric (count of hits)
    """
    return calculate_hits_at_10(recommendations, actual_reorders)


def calculate_basket_coverage(recommendations, actual_reorders):
    """
    Calculate basket coverage: proportion of actual reorders covered by recommendations.

    Args:
        recommendations: List of recommended product IDs
        actual_reorders: List of product IDs actually reordered by user

    Returns:
        Proportion of basket covered (0-1)
    """
    if len(actual_reorders) == 0:
        return 0.0

    rec_set = set(recommendations)
    actual_set = set(actual_reorders)
    hits = len(rec_set & actual_set)

    return hits / len(actual_set)


def paired_wilcoxon_test(metric_baseline, metric_als, alternative='two-sided'):
    """
    Run paired Wilcoxon signed-rank test comparing metrics between two recommenders.

    Args:
        metric_baseline: Array of metrics from baseline recommender
        metric_als: Array of metrics from ALS recommender
        alternative: Type of test ('two-sided', 'greater', 'less')

    Returns:
        Dictionary with test statistic, p-value, and effect sizes
    """
    metric_baseline = np.array(metric_baseline)
    metric_als = np.array(metric_als)

    # Remove pairs with both values zero
    valid_mask = (metric_baseline != 0) | (metric_als != 0)
    metric_baseline = metric_baseline[valid_mask]
    metric_als = metric_als[valid_mask]

    if len(metric_baseline) == 0:
        return {
            'test': 'Wilcoxon Signed-Rank',
            'statistic': np.nan,
            'p_value': np.nan,
            'baseline_mean': 0,
            'als_mean': 0,
            'baseline_median': 0,
            'als_median': 0,
            'valid_pairs': 0
        }

    statistic, p_value = stats.wilcoxon(metric_baseline, metric_als, alternative=alternative)

    return {
        'test': 'Wilcoxon Signed-Rank',
        'statistic': statistic,
        'p_value': p_value,
        'baseline_mean': metric_baseline.mean(),
        'als_mean': metric_als.mean(),
        'baseline_median': np.median(metric_baseline),
        'als_median': np.median(metric_als),
        'baseline_std': metric_baseline.std(),
        'als_std': metric_als.std(),
        'valid_pairs': len(metric_baseline)
    }


def run_paired_comparison_test(user_ids, baseline_recs_dict, als_recs_dict, ground_truth_csr, user_index, product_index):
    """
    Run paired statistical test comparing baseline vs ALS recommendations.

    Args:
        user_ids: Array of user IDs
        baseline_recs_dict: Dict mapping user_id to baseline recommendations
        als_recs_dict: Dict mapping user_id to ALS recommendations
        ground_truth_csr: Sparse matrix of actual reorders for evaluation (e.g., test set)
        user_index: Array mapping user index to user ID
        product_index: Array mapping internal product index to product ID

    Returns:
        Dictionary with metrics and statistical test results
    """
    # Create mapping from user_id to index in ground_truth_csr
    user_id_to_idx = {uid: idx for idx, uid in enumerate(user_index)}

    # Calculate metrics for each user
    baseline_hits = []
    baseline_revenue = []
    baseline_coverage = []

    als_hits = []
    als_revenue = []
    als_coverage = []

    for user_id in user_ids:
        # Get user's actual reorders from the ground_truth_csr
        user_idx = user_id_to_idx.get(user_id)
        if user_idx is None:
            continue

        # Convert internal product indices from ground_truth_csr to actual product IDs
        actual_reorder_internal_indices = ground_truth_csr[user_idx].indices
        actual_reorders = product_index[actual_reorder_internal_indices].tolist()

        # Baseline metrics
        baseline_recs = baseline_recs_dict.get(user_id, [])
        baseline_hits.append(calculate_hits_at_10(baseline_recs, actual_reorders))
        baseline_revenue.append(calculate_revenue_from_hits(baseline_recs, actual_reorders))
        baseline_coverage.append(calculate_basket_coverage(baseline_recs, actual_reorders))

        # ALS metrics
        als_recs = als_recs_dict.get(user_id, [])
        als_hits.append(calculate_hits_at_10(als_recs, actual_reorders))
        als_revenue.append(calculate_revenue_from_hits(als_recs, actual_reorders))
        als_coverage.append(calculate_basket_coverage(als_recs, actual_reorders))

    # Run paired tests for each metric
    hits_test = paired_wilcoxon_test(baseline_hits, als_hits)
    revenue_test = paired_wilcoxon_test(baseline_revenue, als_revenue)
    coverage_test = paired_wilcoxon_test(baseline_coverage, als_coverage)

    return {
        'baseline_hits': baseline_hits,
        'als_hits': als_hits,
        'hits_test': hits_test,
        'baseline_revenue': baseline_revenue,
        'als_revenue': als_revenue,
        'revenue_test': revenue_test,
        'baseline_coverage': baseline_coverage,
        'als_coverage': als_coverage,
        'coverage_test': coverage_test
    }


def load_reorder_matrix_and_indices():
    """Load the saved CSR matrix and indices from data/processed folder."""
    csr_path = "/content/reorder_csr.npz"
    user_index_path = "/content/reorder_user_index.npy"
    product_index_path = "/content/reorder_product_index.npy"

    reorder_csr = sp.load_npz(csr_path).astype(np.float32)
    user_index = np.load(user_index_path)
    product_index = np.load(product_index_path)

    return reorder_csr, user_index, product_index

def create_temporal_split_csr(reorder_csr, test_split_ratio=0.2, random_seed=42):
    """
    Splits the reorder_csr into training and testing CSR matrices using a random split.
    For each user, a percentage of their interactions are randomly assigned to the test set.

    Args:
        reorder_csr: The original sparse matrix of user-item interactions.
        test_split_ratio: The proportion of interactions to put into the test set (0.0 to 1.0).
        random_seed: Random seed for reproducibility.

    Returns:
        train_csr, test_csr: Sparse matrices for training and testing.
    """
    np.random.seed(random_seed)
    num_users, num_items = reorder_csr.shape
    train_rows, train_cols, train_data = [], [], []
    test_rows, test_cols, test_data = [], [], []

    for user_idx in range(num_users):
        user_interactions = reorder_csr.getrow(user_idx).indices
        num_user_interactions = len(user_interactions)

        if num_user_interactions < 2:
            # Users with 0 or 1 interaction are entirely assigned to the training set.
            # They cannot form a meaningful test set with at least one item.
            for item_idx in user_interactions:
                train_rows.append(user_idx)
                train_cols.append(item_idx)
                train_data.append(1) # Assuming binary interactions
            continue # Skip to next user

        # For users with 2 or more interactions, proceed with splitting
        np.random.shuffle(user_interactions)

        # Ensure at least one item in training and at least one item in test
        train_count = max(1, int(num_user_interactions * (1 - test_split_ratio)))
        # If forcing train_count to 1 makes test_count 0, adjust train_count down by 1
        if num_user_interactions - train_count == 0:
            train_count = num_user_interactions - 1
        test_count = num_user_interactions - train_count

        train_interactions = user_interactions[:train_count]
        test_interactions = user_interactions[train_count:]

        # Add to train_csr
        for item_idx in train_interactions:
            train_rows.append(user_idx)
            train_cols.append(item_idx)
            train_data.append(1) # Assuming binary interactions

        # Add to test_csr
        for item_idx in test_interactions:
            test_rows.append(user_idx)
            test_cols.append(item_idx)
            test_data.append(1) # Assuming binary interactions

    train_csr = sp.csr_matrix((train_data, (train_rows, train_cols)), shape=(num_users, num_items), dtype=np.float32)
    test_csr = sp.csr_matrix((test_data, (test_rows, test_cols)), shape=(num_users, num_items), dtype=np.float32)

    return train_csr, test_csr


def get_popularity_based_recommendations(train_csr, product_index, n_recs=10):
    """
    Generate baseline recommendations based on product popularity from the training set.
    """
    # Calculate product popularity (sum of reorders across all users) based on the training data
    product_popularity = np.asarray(train_csr.sum(axis=0)).flatten()

    # Sort products by popularity
    top_products_idx = np.argsort(product_popularity)[::-1][:n_recs]
    top_products = product_index[top_products_idx]

    return top_products.tolist()


def generate_baseline_recommendations(train_csr, product_index, user_index, n_recs=10):
    """
    Generate baseline (popularity-based) recommendations for all users based on training data.
    """
    baseline_recs = {}
    top_products = get_popularity_based_recommendations(train_csr, product_index, n_recs)

    for user_id in user_index:
        baseline_recs[user_id] = top_products

    return baseline_recs


def train_als_model(train_csr):
    """
    Train ALS model on the training reorder matrix.
    """
    alpha = 40
    # The implicit library expects a user-item matrix where rows are users and columns are items.
    # train_csr is already users x products. Create confidence matrix from it.
    confidence_matrix = train_csr.copy()
    confidence_matrix.data = 1 + alpha * confidence_matrix.data

    als = AlternatingLeastSquares(
        factors=64,
        regularization=0.05,
        iterations=30
    )
    # Train directly on the user-item confidence matrix
    # Removed: item_user_matrix = confidence_matrix.T
    als.fit(confidence_matrix) # Pass the confidence_matrix directly, which is users x items

    return als, confidence_matrix


def generate_als_recommendations(train_csr, als_model, user_index, product_index, n_recs=10):
    als_recs = {}

    for user_idx, user_id in enumerate(user_index):
        # Get the items the user has already interacted with from the TRAIN CSR matrix
        # These are internal product indices for the current user_idx
        user_items_internal_indices = train_csr[user_idx]

        # Use the implicit.recommend method, which automatically filters already-liked items
        recommended_internal_indices, _ = als_model.recommend(
            userid=user_idx,
            user_items=user_items_internal_indices, # Pass the sparse row for the user from TRAIN_CSR
            N=n_recs,
            filter_already_liked_items=True # Explicitly ensure already-liked items are filtered
        )

        # Convert internal item indices back to original product IDs
        recommended_product_ids = product_index[recommended_internal_indices].tolist()
        als_recs[user_id] = recommended_product_ids

    return als_recs


In [ ]:
print("Loading reorder matrix and indices...")
reorder_csr, user_index, product_index = load_reorder_matrix_and_indices()

print(f"Loaded full matrix shape: {reorder_csr.shape}")
print(f"Number of users: {len(user_index)}")
print(f"Number of products: {len(product_index)}")

print("\nSplitting data into training and test sets...")
train_csr, test_csr = create_temporal_split_csr(reorder_csr, test_split_ratio=0.2)
print(f"Train matrix shape: {train_csr.shape}")
print(f"Test matrix shape: {test_csr.shape}")

# Calculate and print average actual reorders per user in the test set
num_users_in_test_set = test_csr.getnnz(axis=1).astype(bool).sum()
if num_users_in_test_set > 0:
    avg_test_reorders = test_csr.sum() / num_users_in_test_set
    print(f"Average actual reorders per user in test set (for evaluated users): {avg_test_reorders:.2f}")
else:
    print("No users in the test set with actual reorders.")

print("\n" + "="*80)
print("GENERATING RECOMMENDATIONS")
print("="*80)

print("\nGenerating popularity-based recommendations for all users (based on training data)...")
baseline_recs_dict = generate_baseline_recommendations(
    train_csr, product_index, user_index, n_recs=10
)
print(f"✓ Generated baseline recommendations for {len(baseline_recs_dict)} users")

print("\nTraining ALS model on training data...")
als_model, confidence_matrix = train_als_model(train_csr)

print("Generating ALS recommendations for all users (filtering items seen in training data)...")
als_recs_dict = generate_als_recommendations(
    train_csr, als_model, user_index, product_index, n_recs=10
)
print(f"✓ Generated ALS recommendations for {len(als_recs_dict)} users")

print("\n" + "="*80)
print("PAIRED STATISTICAL TEST")
print("="*80)

print("\nCalculating metrics and running paired Wilcoxon signed-rank tests (evaluating against test set)...")
results = run_paired_comparison_test(user_index, baseline_recs_dict, als_recs_dict,
                                    test_csr, user_index, product_index)

print("\n" + "-"*80)
print("HITS@10 TEST")
print("-"*80)
print(f"Baseline (Popularity):")
print(f"  Mean: {results['hits_test']['baseline_mean']:.4f}")
print(f"  Median: {results['hits_test']['baseline_median']:.4f}")
print(f"  Std Dev: {results['hits_test']['baseline_std']:.4f}")

print(f"\nALS Recommender:")
print(f"  Mean: {results['hits_test']['als_mean']:.4f}")
print(f"  Median: {results['hits_test']['als_median']:.4f}")
print(f"  Std Dev: {results['hits_test']['als_std']:.4f}")

print(f"\nPaired Wilcoxon Test (n={results['hits_test']['valid_pairs']} pairs):")
print(f"  Test Statistic: {results['hits_test']['statistic']:.2f}")
print(f"  P-value: {results['hits_test']['p_value']:.6f}")
if results['hits_test']['p_value'] < 0.05:
    print(f"  ✓ SIGNIFICANT (p < 0.05)")
    if results['hits_test']['als_mean'] > results['hits_test']['baseline_mean']:
        print(f"    ALS performs BETTER than Popularity")
    else:
        print(f"    Popularity performs BETTER than ALS")
else:
    print(f"  ✗ NOT SIGNIFICANT (p \u2265 0.05)")

print("\n" + "-"*80)
print("REVENUE FROM HITS TEST")
print("-"*80)
print(f"Baseline (Popularity):")
print(f"  Mean: {results['revenue_test']['baseline_mean']:.4f}")
print(f"  Median: {results['revenue_test']['baseline_median']:.4f}")
print(f"  Std Dev: {results['revenue_test']['baseline_std']:.4f}")

print(f"\nALS Recommender:")
print(f"  Mean: {results['revenue_test']['als_mean']:.4f}")
print(f"  Median: {results['revenue_test']['als_median']:.4f}")
print(f"  Std Dev: {results['revenue_test']['als_std']:.4f}")

print(f"\nPaired Wilcoxon Test (n={results['revenue_test']['valid_pairs']} pairs):")
print(f"  Test Statistic: {results['revenue_test']['statistic']:.2f}")
print(f"  P-value: {results['revenue_test']['p_value']:.6f}")
if results['revenue_test']['p_value'] < 0.05:
    print(f"  ✓ SIGNIFICANT (p < 0.05)")
    if results['revenue_test']['als_mean'] > results['revenue_test']['baseline_mean']:
        print(f"    ALS generates MORE revenue from hits")
    else:
        print(f"    Popularity generates MORE revenue from hits")
else:
    print(f"  ✗ NOT SIGNIFICANT (p \u2265 0.05)")

print("\n" + "-"*80)
print("BASKET COVERAGE TEST")
print("-"*80)
print(f"Baseline (Popularity):")
print(f"  Mean: {results['coverage_test']['baseline_mean']:.4f}")
print(f"  Median: {results['coverage_test']['baseline_median']:.4f}")
print(f"  Std Dev: {results['coverage_test']['baseline_std']:.4f}")

print(f"\nALS Recommender:")
print(f"  Mean: {results['coverage_test']['als_mean']:.4f}")
print(f"  Median: {results['coverage_test']['als_median']:.4f}")
print(f"  Std Dev: {results['coverage_test']['als_std']:.4f}")

print(f"\nPaired Wilcoxon Test (n={results['coverage_test']['valid_pairs']} pairs):")
print(f"  Test Statistic: {results['coverage_test']['statistic']:.2f}")
print(f"  P-value: {results['coverage_test']['p_value']:.6f}")
if results['coverage_test']['p_value'] < 0.05:
    print(f"  ✓ SIGNIFICANT (p < 0.05)")
    if results['coverage_test']['als_mean'] > results['coverage_test']['baseline_mean']:
        print(f"    ALS provides BETTER basket coverage")
    else:
        print(f"    Popularity provides BETTER basket coverage")
else:
    print(f"  ✗ NOT SIGNIFICANT (p \u2265 0.05)")

print("\n" + "="*80)

Loading reorder matrix and indices...
Loaded full matrix shape: (203026, 25620)
Number of users: 203026
Number of products: 25620

Splitting data into training and test sets...
Train matrix shape: (203026, 25620)
Test matrix shape: (203026, 25620)
Average actual reorders per user in test set (for evaluated users): 5.76

GENERATING RECOMMENDATIONS

Generating popularity-based recommendations for all users (based on training data)...
✓ Generated baseline recommendations for 203026 users

Training ALS model on training data...


  0%|          | 0/30 [00:00<?, ?it/s]

Generating ALS recommendations for all users (filtering items seen in training data)...
✓ Generated ALS recommendations for 203026 users

PAIRED STATISTICAL TEST

Calculating metrics and running paired Wilcoxon signed-rank tests (evaluating against test set)...

--------------------------------------------------------------------------------
HITS@10 TEST
--------------------------------------------------------------------------------
Baseline (Popularity):
  Mean: 0.7792
  Median: 1.0000
  Std Dev: 0.7398

ALS Recommender:
  Mean: 1.0218
  Median: 1.0000
  Std Dev: 0.7942

Paired Wilcoxon Test (n=91327 pairs):
  Test Statistic: 898650375.50
  P-value: 0.000000
  ✓ SIGNIFICANT (p < 0.05)
    ALS performs BETTER than Popularity

--------------------------------------------------------------------------------
REVENUE FROM HITS TEST
--------------------------------------------------------------------------------
Baseline (Popularity):
  Mean: 0.7792
  Median: 1.0000
  Std Dev: 0.7398

ALS 

In [ ]:
print("Debugging generate_als_recommendations for potential IndexError...")

# Verify product_index size
max_valid_product_index = len(product_index) - 1
print(f"Max valid index for product_index: {max_valid_product_index} (total products: {len(product_index)})")

problem_found = False
# Check a sample of users to quickly identify if the issue is systematic.
# We'll pick 100 random users or all users if there are fewer than 100.
sample_size = min(100, len(user_index))
sample_user_indices_for_debug = np.random.choice(len(user_index), size=sample_size, replace=False)

for user_idx_debug in sample_user_indices_for_debug:
    user_id_debug = user_index[user_idx_debug]
    user_items_internal_indices_debug = reorder_csr[user_idx_debug]

    try:
        # Call als_model.recommend
        recommended_internal_indices_debug, _ = als_model.recommend(
            userid=user_idx_debug,
            user_items=user_items_internal_indices_debug,
            N=10, # Using 10 as per the original call
            filter_already_liked_items=True
        )

        if len(recommended_internal_indices_debug) > 0:
            max_rec_idx = np.max(recommended_internal_indices_debug)
            min_rec_idx = np.min(recommended_internal_indices_debug)

            # Check for out-of-bounds indices
            if max_rec_idx > max_valid_product_index or min_rec_idx < 0:
                print(f"\nERROR: Out-of-bounds internal index found for user_idx {user_idx_debug} (user_id {user_id_debug})")
                print(f"  Recommended internal indices: {recommended_internal_indices_debug}")
                print(f"  Max recommended index: {max_rec_idx}, Min recommended index: {min_rec_idx}")
                problem_found = True
                break # Found the problem, stop and report

        # Attempt the product_index lookup to catch any subtle issues
        _ = product_index[recommended_internal_indices_debug].tolist()

    except IndexError as e:
        print(f"\nIndexError encountered for user_idx {user_idx_debug} (user_id {user_id_debug}) during product_index lookup: {e}")
        if 'recommended_internal_indices_debug' in locals():
            print(f"  Recommended internal indices: {recommended_internal_indices_debug}")
        problem_found = True
        break # Found the problem, stop and report
    except Exception as e:
        print(f"\nUnexpected error for user_idx {user_idx_debug} (user_id {user_id_debug}): {type(e).__name__} - {e}")
        problem_found = True
        break

if not problem_found:
    print(f"\nNo obvious IndexError found for the {sample_size} sampled users in generate_als_recommendations.")
    print("The issue might be very specific to certain users not covered by the sample, or it might be occurring in the 'run_paired_comparison_test' function itself.")

# You can extend this to check all users if needed, but it might take longer:
# if not problem_found:
#     print("Checking all users for potential IndexError (this may take a while)...")
#     # You would repeat the loop above for all user_idx in range(len(user_index))


Debugging generate_als_recommendations for potential IndexError...
Max valid index for product_index: 25619 (total products: 25620)

No obvious IndexError found for the 100 sampled users in generate_als_recommendations.
The issue might be very specific to certain users not covered by the sample, or it might be occurring in the 'run_paired_comparison_test' function itself.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math

# Check if CUDA is available and set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print('\n' + '='*80)
print('PREPARING DATA FOR SASREC (PyTorch)')
print('='*80)

# --- Data Loading (moved from a previous cell to ensure variables are defined) ---
print("Loading reorder matrix and indices...")
reorder_csr, user_index, product_index = load_reorder_matrix_and_indices()
print(f"Loaded full matrix shape: {reorder_csr.shape}")
print(f"Number of users: {len(user_index)}")
print(f"Number of products: {len(product_index)}")

print("Splitting data into training and test sets...")
train_csr, test_csr = create_temporal_split_csr(reorder_csr, test_split_ratio=0.2)
print(f"Train matrix shape: {train_csr.shape}")
print(f"Test matrix shape: {test_csr.shape}")
# --- End Data Loading ---

MAX_SEQ_LEN = 50 # Define max sequence length for SASRec
BATCH_SIZE = 256 # Batch size for PyTorch DataLoaders

train_loader, val_loader, test_loader, item_to_id, id_to_item, num_unique_items, user_sequences_map_for_recs = create_sequential_data(
    reorder_csr, user_index, product_index, max_seq_len=MAX_SEQ_LEN, batch_size=BATCH_SIZE
)

print(f"Number of batches in training data: {len(train_loader)}")
print(f"Number of batches in validation data: {len(val_loader)}")
print(f"Number of batches in test data: {len(test_loader)}")

print('\n' + '='*80)
print('TRAINING SASREC MODEL (PyTorch)')
print('='*80)

# Model parameters
d_model = 64
num_heads = 2
num_blocks = 2
dropout_rate = 0.2
NUM_EPOCHS = 10 # You might need to adjust epochs
LEARNING_RATE = 0.001

sasrec_pytorch_model = SASRecPyTorch(
    num_items=num_unique_items,
    max_seq_len=MAX_SEQ_LEN,
    d_model=d_model,
    num_heads=num_heads,
    num_blocks=num_blocks,
    dropout_rate=dropout_rate
).to(device)

optimizer = optim.Adam(sasrec_pytorch_model.parameters(), lr=LEARNING_RATE)
# Ignore padding_idx=0 in loss calculation
criterion = nn.CrossEntropyLoss(ignore_index=0)

# Training loop
print("Starting SASRec PyTorch model training...")
for epoch in range(NUM_EPOCHS):
    sasrec_pytorch_model.train() # Set model to training mode
    total_loss = 0
    for batch_idx, (sequences, targets) in enumerate(train_loader):
        sequences, targets = sequences.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = sasrec_pytorch_model(sequences)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader);

    # Validation step
    sasrec_pytorch_model.eval() # Set model to evaluation mode
    val_loss = 0
    correct_predictions = 0
    total_targets = 0
    with torch.no_grad():
        for sequences, targets in val_loader:
            sequences, targets = sequences.to(device), targets.to(device)
            outputs = sasrec_pytorch_model(sequences)
            loss = criterion(outputs, targets)
            val_loss += loss.item()

            # Calculate accuracy (excluding padding targets)
            _, predicted = torch.max(outputs.data, 1);
            valid_targets_mask = (targets != 0)
            correct_predictions += (predicted[valid_targets_mask] == targets[valid_targets_mask]).sum().item()
            total_targets += valid_targets_mask.sum().item()

    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = correct_predictions / total_targets if total_targets > 0 else 0

    print(f'Epoch {epoch+1}/{NUM_EPOCHS}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}')
print("SASRec PyTorch model training complete.")

print('\n' + '='*80)
print('GENERATING SASREC RECOMMENDATIONS (PyTorch)')
print('='*80)

sasrec_recs_dict = generate_sasrec_recommendations(
    sasrec_pytorch_model, user_sequences_map_for_recs, user_index, item_to_id, id_to_item, MAX_SEQ_LEN, n_recs=10,
    train_csr=train_csr, product_index_global=product_index # Pass train_csr and product_index for filtering
)
print(f"✓ Generated SASRec recommendations for {len(sasrec_recs_dict)} users")

print('\n' + '='*80)
print('PAIRED STATISTICAL TEST (SASRec vs Baseline)')
print('='*80)

sasrec_results_vs_baseline = run_paired_comparison_test(
    user_index, baseline_recs_dict, sasrec_recs_dict,
    test_csr, user_index, product_index # Changed reorder_csr to test_csr for proper evaluation
)

print('\n' + '-'*80)
print('HITS@10 TEST (SASRec vs Baseline)')
print('-'*80)
print(f"Baseline (Popularity):")
print(f"  Mean: {sasrec_results_vs_baseline['hits_test']['baseline_mean']:.4f}")
print(f"  Median: {sasrec_results_vs_baseline['hits_test']['baseline_median']:.4f}")
print(f"  Std Dev: {sasrec_results_vs_baseline['hits_test']['baseline_std']:.4f}")

print(f"\nSASRec Recommender:")
print(f"  Mean: {sasrec_results_vs_baseline['hits_test']['als_mean']:.4f}")
print(f"  Median: {sasrec_results_vs_baseline['hits_test']['als_median']:.4f}")
print(f"  Std Dev: {sasrec_results_vs_baseline['hits_test']['als_std']:.4f}")

print(f"\nPaired Wilcoxon Test (n={sasrec_results_vs_baseline['hits_test']['valid_pairs']} pairs):")
print(f"  Test Statistic: {sasrec_results_vs_baseline['hits_test']['statistic']:.2f}")
print(f"  P-value: {sasrec_results_vs_baseline['hits_test']['p_value']:.6f}")
if sasrec_results_vs_baseline['hits_test']['p_value'] < 0.05:
    print(f"  ✓ SIGNIFICANT (p < 0.05)")
    if sasrec_results_vs_baseline['hits_test']['als_mean'] > sasrec_results_vs_baseline['hits_test']['baseline_mean']:
        print(f"    SASRec performs BETTER than Popularity")
    else:
        print(f"    Popularity performs BETTER than SASRec")
else:
    print(f"  ✗ NOT SIGNIFICANT (p \u2265 0.05)")

print('\n' + '-'*80)
print('REVENUE FROM HITS TEST (SASRec vs Baseline)')
print('-'*80)
print(f"Baseline (Popularity):")
print(f"  Mean: {sasrec_results_vs_baseline['revenue_test']['baseline_mean']:.4f}")
print(f"  Median: {sasrec_results_vs_baseline['revenue_test']['baseline_median']:.4f}")
print(f"  Std Dev: {sasrec_results_vs_baseline['revenue_test']['baseline_std']:.4f}")

print(f"\nSASRec Recommender:")
print(f"  Mean: {sasrec_results_vs_baseline['revenue_test']['als_mean']:.4f}")
print(f"  Median: {sasrec_results_vs_baseline['revenue_test']['als_median']:.4f}")
print(f"  Std Dev: {sasrec_results_vs_baseline['revenue_test']['als_std']:.4f}")

print(f"\nPaired Wilcoxon Test (n={sasrec_results_vs_baseline['revenue_test']['valid_pairs']} pairs):")
print(f"  Test Statistic: {sasrec_results_vs_baseline['revenue_test']['statistic']:.2f}")
print(f"  P-value: {sasrec_results_vs_baseline['revenue_test']['p_value']:.6f}")
if sasrec_results_vs_baseline['revenue_test']['p_value'] < 0.05:
    print(f"  ✓ SIGNIFICANT (p < 0.05)")
    if sasrec_results_vs_baseline['revenue_test']['als_mean'] > sasrec_results_vs_baseline['revenue_test']['baseline_mean']:
        print(f"    SASRec generates MORE revenue from hits")
    else:
        print(f"    Popularity generates MORE revenue from hits")
else:
    print(f"  ✗ NOT SIGNIFICANT (p \u2265 0.05)")

print('\n' + '-'*80)
print('BASKET COVERAGE TEST (SASRec vs Baseline)')
print('-'*80)
print(f"Baseline (Popularity):")
print(f"  Mean: {sasrec_results_vs_baseline['coverage_test']['baseline_mean']:.4f}")
print(f"  Median: {sasrec_results_vs_baseline['coverage_test']['baseline_median']:.4f}")
print(f"  Std Dev: {sasrec_results_vs_baseline['coverage_test']['baseline_std']:.4f}")

print(f"\nSASRec Recommender:")
print(f"  Mean: {sasrec_results_vs_baseline['coverage_test']['als_mean']:.4f}")
print(f"  Median: {sasrec_results_vs_baseline['coverage_test']['als_median']:.4f}")
print(f"  Std Dev: {sasrec_results_vs_baseline['coverage_test']['als_std']:.4f}")

print(f"\nPaired Wilcoxon Test (n={sasrec_results_vs_baseline['coverage_test']['valid_pairs']} pairs):")
print(f"  Test Statistic: {sasrec_results_vs_baseline['coverage_test']['statistic']:.2f}")
print(f"  P-value: {sasrec_results_vs_baseline['coverage_test']['p_value']:.6f}")
if sasrec_results_vs_baseline['coverage_test']['p_value'] < 0.05:
    print(f"  ✓ SIGNIFICANT (p < 0.05)")
    if sasrec_results_vs_baseline['coverage_test']['als_mean'] > sasrec_results_vs_baseline['coverage_test']['baseline_mean']:
        print(f"    SASRec provides BETTER basket coverage")
    else:
        print(f"    Popularity provides BETTER basket coverage")
else:
    print(f"  ✗ NOT SIGNIFICANT (p \u2265 0.05)")

print('\n' + '='*80)


Using device: cuda

PREPARING DATA FOR SASREC (PyTorch)
Loading reorder matrix and indices...
Loaded full matrix shape: (203026, 25620)
Number of users: 203026
Number of products: 25620
Splitting data into training and test sets...
Train matrix shape: (203026, 25620)
Test matrix shape: (203026, 25620)
Total unique items for SASRec: 25620
Total sequences created: 5049477
Number of batches in training data: 15780
Number of batches in validation data: 1973
Number of batches in test data: 1973

TRAINING SASREC MODEL (PyTorch)
Starting SASRec PyTorch model training...
Epoch 1/10, Train Loss: 6.4671, Val Loss: 5.7727, Val Accuracy: 0.0832
Epoch 2/10, Train Loss: 5.7710, Val Loss: 5.5094, Val Accuracy: 0.0914
Epoch 3/10, Train Loss: 5.5725, Val Loss: 5.3955, Val Accuracy: 0.0946
Epoch 4/10, Train Loss: 5.4691, Val Loss: 5.3428, Val Accuracy: 0.0974
Epoch 5/10, Train Loss: 5.4035, Val Loss: 5.2993, Val Accuracy: 0.0989
Epoch 6/10, Train Loss: 5.3558, Val Loss: 5.2680, Val Accuracy: 0.1004
Epoc

## A/B Test: ALS vs SASRec

Finally, let's compare the performance of the ALS recommender against the SASRec recommender using paired statistical tests for Hits@10, Revenue from Hits, and Basket Coverage. This will help us understand which model performs better on these key metrics.

In [ ]:
print('\n' + '='*80)
print('PAIRED STATISTICAL TEST (ALS vs SASRec)')
print('='*80)

print('\nCalculating metrics and running paired Wilcoxon signed-rank tests for ALS vs SASRec...')
als_vs_sasrec_results = run_paired_comparison_test(
    user_index, als_recs_dict, sasrec_recs_dict,
    test_csr, user_index, product_index # Changed reorder_csr to test_csr for proper evaluation
)

print('\n' + '-'*80)
print('HITS@10 TEST (ALS vs SASRec)')
print('-'*80)
print(f"ALS Recommender:")
print(f"  Mean: {als_vs_sasrec_results['hits_test']['baseline_mean']:.4f}")
print(f"  Median: {als_vs_sasrec_results['hits_test']['baseline_median']:.4f}")
print(f"  Std Dev: {als_vs_sasrec_results['hits_test']['baseline_std']:.4f}")

print(f"\nSASRec Recommender:")
print(f"  Mean: {als_vs_sasrec_results['hits_test']['als_mean']:.4f}")
print(f"  Median: {als_vs_sasrec_results['hits_test']['als_median']:.4f}")
print(f"  Std Dev: {als_vs_sasrec_results['hits_test']['als_std']:.4f}")

print(f"\nPaired Wilcoxon Test (n={als_vs_sasrec_results['hits_test']['valid_pairs']} pairs):")
print(f"  Test Statistic: {als_vs_sasrec_results['hits_test']['statistic']:.2f}")
print(f"  P-value: {als_vs_sasrec_results['hits_test']['p_value']:.6f}")
if als_vs_sasrec_results['hits_test']['p_value'] < 0.05:
    print(f"  ✓ SIGNIFICANT (p < 0.05)")
    if als_vs_sasrec_results['hits_test']['als_mean'] > als_vs_sasrec_results['hits_test']['baseline_mean']:
        print(f"    SASRec performs BETTER than ALS")
    else:
        print(f"    ALS performs BETTER than SASRec")
else:
    print(f"  ✗ NOT SIGNIFICANT (p \u2265 0.05)")

print('\n' + '-'*80)
print('REVENUE FROM HITS TEST (ALS vs SASRec)')
print('-'*80)
print(f"ALS Recommender:")
print(f"  Mean: {als_vs_sasrec_results['revenue_test']['baseline_mean']:.4f}")
print(f"  Median: {als_vs_sasrec_results['revenue_test']['baseline_median']:.4f}")
print(f"  Std Dev: {als_vs_sasrec_results['revenue_test']['baseline_std']:.4f}")

print(f"\nSASRec Recommender:")
print(f"  Mean: {als_vs_sasrec_results['revenue_test']['als_mean']:.4f}")
print(f"  Median: {als_vs_sasrec_results['revenue_test']['als_median']:.4f}")
print(f"  Std Dev: {als_vs_sasrec_results['revenue_test']['als_std']:.4f}")

print(f"\nPaired Wilcoxon Test (n={als_vs_sasrec_results['revenue_test']['valid_pairs']} pairs):")
print(f"  Test Statistic: {als_vs_sasrec_results['revenue_test']['statistic']:.2f}")
print(f"  P-value: {als_vs_sasrec_results['revenue_test']['p_value']:.6f}")
if als_vs_sasrec_results['revenue_test']['p_value'] < 0.05:
    print(f"  ✓ SIGNIFICANT (p < 0.05)")
    if als_vs_sasrec_results['revenue_test']['als_mean'] > als_vs_sasrec_results['revenue_test']['baseline_mean']:
        print(f"    SASRec generates MORE revenue from hits")
    else:
        print(f"    ALS generates MORE revenue from hits")
else:
    print(f"  ✗ NOT SIGNIFICANT (p \u2265 0.05)")

print('\n' + '-'*80)
print('BASKET COVERAGE TEST (ALS vs SASRec)')
print('-'*80)
print(f"ALS Recommender:")
print(f"  Mean: {als_vs_sasrec_results['coverage_test']['baseline_mean']:.4f}")
print(f"  Median: {als_vs_sasrec_results['coverage_test']['baseline_median']:.4f}")
print(f"  Std Dev: {als_vs_sasrec_results['coverage_test']['baseline_std']:.4f}")

print(f"\nSASRec Recommender:")
print(f"  Mean: {als_vs_sasrec_results['coverage_test']['als_mean']:.4f}")
print(f"  Median: {als_vs_sasrec_results['coverage_test']['als_median']:.4f}")
print(f"  Std Dev: {als_vs_sasrec_results['coverage_test']['als_std']:.4f}")

print(f"\nPaired Wilcoxon Test (n={als_vs_sasrec_results['coverage_test']['valid_pairs']} pairs):")
print(f"  Test Statistic: {als_vs_sasrec_results['coverage_test']['statistic']:.2f}")
print(f"  P-value: {als_vs_sasrec_results['coverage_test']['p_value']:.6f}")
if als_vs_sasrec_results['coverage_test']['p_value'] < 0.05:
    print(f"  ✓ SIGNIFICANT (p < 0.05)")
    if als_vs_sasrec_results['coverage_test']['als_mean'] > als_vs_sasrec_results['coverage_test']['baseline_mean']:
        print(f"    SASRec provides BETTER basket coverage")
    else:
        print(f"    ALS provides BETTER basket coverage")
else:
    print(f"  ✗ NOT SIGNIFICANT (p \u2265 0.05)")

print('\n' + '='*80)



PAIRED STATISTICAL TEST (ALS vs SASRec)

Calculating metrics and running paired Wilcoxon signed-rank tests for ALS vs SASRec...

--------------------------------------------------------------------------------
HITS@10 TEST (ALS vs SASRec)
--------------------------------------------------------------------------------
ALS Recommender:
  Mean: 1.2491
  Median: 1.0000
  Std Dev: 0.6979

SASRec Recommender:
  Mean: 0.1041
  Median: 0.0000
  Std Dev: 0.3102

Paired Wilcoxon Test (n=74705 pairs):
  Test Statistic: 140131322.00
  P-value: 0.000000
  ✓ SIGNIFICANT (p < 0.05)
    ALS performs BETTER than SASRec

--------------------------------------------------------------------------------
REVENUE FROM HITS TEST (ALS vs SASRec)
--------------------------------------------------------------------------------
ALS Recommender:
  Mean: 1.2491
  Median: 1.0000
  Std Dev: 0.6979

SASRec Recommender:
  Mean: 0.1041
  Median: 0.0000
  Std Dev: 0.3102

Paired Wilcoxon Test (n=74705 pairs):
  Test St

## Creating a Streamlit Dashboard

To present these results in an interactive and user-friendly way, we can create a Streamlit dashboard. This involves:

1.  Installing Streamlit.
2.  Saving the calculated comparison results (`results`, `sasrec_results_vs_baseline`, `als_vs_sasrec_results`) to a JSON file so the Streamlit app can load them.
3.  Writing the Streamlit application code (`app.py`) that loads the results and displays them.
4.  Running the Streamlit application.

In [ ]:
# Install Streamlit
!pip install streamlit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 115.1 MB/s eta 0:00:00


In [ ]:
import json

# Combine all results into a single dictionary for easy saving
all_results = {
    'als_vs_baseline': {
        'hits_test': results['hits_test'],
        'revenue_test': results['revenue_test'],
        'coverage_test': results['coverage_test'],
    },
    'sasrec_vs_baseline': {
        'hits_test': sasrec_results_vs_baseline['hits_test'],
        'revenue_test': sasrec_results_vs_baseline['revenue_test'],
        'coverage_test': sasrec_results_vs_baseline['coverage_test'],
    },
    'als_vs_sasrec': {
        'hits_test': als_vs_sasrec_results['hits_test'],
        'revenue_test': als_vs_sasrec_results['revenue_test'],
        'coverage_test': als_vs_sasrec_results['coverage_test'],
    },
}

# Convert numpy types to native Python types for JSON serialization
def convert_numpy_to_native(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_numpy_to_native(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy_to_native(elem) for elem in obj]
    return obj

all_results_native = convert_numpy_to_native(all_results)

# Save the results to a JSON file
with open('recommendation_ab_test_results.json', 'w') as f:
    json.dump(all_results_native, f, indent=4)

print("Results saved to 'recommendation_ab_test_results.json'")

Results saved to 'recommendation_ab_test_results.json'


Here's the code for the Streamlit application. It will load the `recommendation_ab_test_results.json` file and display the metrics and A/B test outcomes for each comparison.

After executing the cell below, a file named `app.py` will be created in your Colab environment.

In [ ]:
%%writefile app.py
import streamlit as st
import json
import pandas as pd

def display_test_results(title, test_results, recommender1_name, recommender2_name):
    st.subheader(title)

    metrics_data = []
    for metric_name, data in test_results.items():
        metrics_data.append({
            'Metric': metric_name.replace('_test', '').replace('_', ' ').title(),
            f'{recommender1_name} Mean': f"{data['baseline_mean']:.4f}",
            f'{recommender1_name} Median': f"{data['baseline_median']:.4f}",
            f'{recommender1_name} Std Dev': f"{data['baseline_std']:.4f}",
            f'{recommender2_name} Mean': f"{data['als_mean']:.4f}", # 'als_mean' is used generically in the test function
            f'{recommender2_name} Median': f"{data['als_median']:.4f}",
            f'{recommender2_name} Std Dev': f"{data['als_std']:.4f}",
            'P-value': f"{data['p_value']:.6f}",
            'Significance': '✓ SIGNIFICANT' if data['p_value'] < 0.05 else '✗ NOT SIGNIFICANT',
            'Better Performer': (
                f'{recommender2_name}' if data['als_mean'] > data['baseline_mean'] else f'{recommender1_name}'
            ) if data['p_value'] < 0.05 else 'No significant difference'
        })
    st.table(pd.DataFrame(metrics_data).set_index('Metric'))


st.set_page_config(layout="wide", page_title="Recommendation System A/B Test Results")
st.title("Recommendation System A/B Test Results")

# Load results from JSON
try:
    with open('recommendation_ab_test_results.json', 'r') as f:
        all_results = json.load(f)
except FileNotFoundError:
    st.error("Error: 'recommendation_ab_test_results.json' not found. Please run the previous cells to generate the results file.")
    st.stop()

# Display results for ALS vs Baseline
display_test_results(
    "ALS vs Baseline (Popularity)",
    all_results['als_vs_baseline'],
    "Popularity",
    "ALS"
)

# Display results for SASRec vs Baseline
display_test_results(
    "SASRec vs Baseline (Popularity)",
    all_results['sasrec_vs_baseline'],
    "Popularity",
    "SASRec"
)

# Display results for ALS vs SASRec
display_test_results(
    "ALS vs SASRec",
    all_results['als_vs_sasrec'],
    "ALS",
    "SASRec"
)

st.markdown("--- ")
st.info("A significant p-value (typically < 0.05) suggests a statistically significant difference between the two recommenders for that metric.")


Writing app.py


### Integrating Churn and Customer Lifetime Value (CLTV)

To accurately calculate and predict **Churn Rate** and **Customer Lifetime Value (CLTV)**, we would typically require more granular transaction data, including:

*   **Timestamps for each user interaction/purchase:** Essential for defining activity periods, inactivity periods (for churn), and time-based revenue calculations (for CLTV).
*   **Monetary value of each transaction:** Crucial for calculating CLTV.
*   **User registration/start dates:** Helps establish a user's entire lifecycle.

The current `reorder_csr` matrix provides a snapshot of *products reordered* by users, but lacks the specific temporal and monetary information needed for a robust calculation of these metrics.

For the purpose of demonstrating integration into the project's reporting and A/B testing framework, we will simulate these metrics as overall project statistics. In a real-world scenario, a dedicated data pipeline would calculate these values based on actual customer behavior data.

In [ ]:
import random

# --- Simulate Churn and CLTV Metrics ---
# In a real scenario, these would be calculated from detailed transaction logs
# For demonstration, we'll generate some plausible synthetic values.

def calculate_dummy_churn_metrics(num_users):
    # Simulate an overall churn rate
    churn_rate = random.uniform(0.15, 0.35) # e.g., between 15% and 35%
    num_churned_users = int(num_users * churn_rate)

    return {
        'overall_churn_rate': churn_rate,
        'num_churned_users': num_churned_users,
        'total_users_considered': num_users
    }

def calculate_dummy_cltv_metrics(num_users):
    # Simulate an overall average CLTV
    avg_cltv = random.uniform(500.0, 1500.0) # e.g., between $500 and $1500
    cltv_std = avg_cltv * random.uniform(0.3, 0.6) # Standard deviation

    return {
        'overall_average_cltv': avg_cltv,
        'cltv_std_dev': cltv_std,
        'total_users_considered': num_users
    }

# Assuming 'user_index' represents all relevant users in our dataset
num_total_users = len(user_index) # From earlier data loading

consumer_churn_metrics = calculate_dummy_churn_metrics(num_total_users)
consumer_cltv_metrics = calculate_dummy_cltv_metrics(num_total_users)

print("Simulated Churn Metrics:")
for k, v in consumer_churn_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

print("\nSimulated CLTV Metrics:")
for k, v in consumer_cltv_metrics.items():
    print(f"  {k}: {v:.2f}" if isinstance(v, float) else f"  {k}: {v}")


# --- Update the results JSON to include these new metrics ---
# Add the new consumer metrics to the 'all_results' dictionary
all_results_native['consumer_metrics'] = {
    'churn': consumer_churn_metrics,
    'cltv': consumer_cltv_metrics
}

# Resave the updated results to the JSON file
with open('recommendation_ab_test_results.json', 'w') as f:
    json.dump(all_results_native, f, indent=4)

print("\nUpdated 'recommendation_ab_test_results.json' with consumer metrics.")

Simulated Churn Metrics:
  overall_churn_rate: 0.2951
  num_churned_users: 59910
  total_users_considered: 203026

Simulated CLTV Metrics:
  overall_average_cltv: 1029.32
  cltv_std_dev: 598.20
  total_users_considered: 203026

Updated 'recommendation_ab_test_results.json' with consumer metrics.


### Extending the Streamlit Dashboard to Display Consumer Metrics

Now that we've simulated (or in a real scenario, calculated) churn and CLTV metrics and included them in our `recommendation_ab_test_results.json` file, we can modify the `app.py` script to display these alongside the recommendation system A/B test results. This will provide a more holistic view of the project's impact on key business indicators.

Note that to truly assess the *impact* of a recommender system on churn or CLTV, a long-running A/B test would be needed where different user groups are exposed to different recommendation strategies, and their subsequent churn/CLTV behavior is tracked.

In [ ]:
%%writefile app.py
import streamlit as st
import json
import pandas as pd

def display_test_results(title, test_results, recommender1_name, recommender2_name):
    st.subheader(title)

    metrics_data = []
    for metric_name, data in test_results.items():
        metrics_data.append({
            'Metric': metric_name.replace('_test', '').replace('_', ' ').title(),
            f'{recommender1_name} Mean': f"{data['baseline_mean']:.4f}",
            f'{recommender1_name} Median': f"{data['baseline_median']:.4f}",
            f'{recommender1_name} Std Dev': f"{data['baseline_std']:.4f}",
            f'{recommender2_name} Mean': f"{data['als_mean']:.4f}", # 'als_mean' is used generically in the test function
            f'{recommender2_name} Median': f"{data['als_median']:.4f}",
            f'{recommender2_name} Std Dev': f"{data['als_std']:.4f}",
            'P-value': f"{data['p_value']:.6f}",
            'Significance': '✓ SIGNIFICANT' if data['p_value'] < 0.05 else '✗ NOT SIGNIFICANT',
            'Better Performer': (
                f'{recommender2_name}' if data['als_mean'] > data['baseline_mean'] else f'{recommender1_name}'
            ) if data['p_value'] < 0.05 else 'No significant difference'
        })
    st.table(pd.DataFrame(metrics_data).set_index('Metric'))


st.set_page_config(layout="wide", page_title="Recommendation System A/B Test Results")
st.title("Recommendation System A/B Test Results")

# Load results from JSON
try:
    with open('recommendation_ab_test_results.json', 'r') as f:
        all_results = json.load(f)
except FileNotFoundError:
    st.error("Error: 'recommendation_ab_test_results.json' not found. Please run the previous cells to generate the results file.")
    st.stop()

# Display results for ALS vs Baseline
display_test_results(
    "ALS vs Baseline (Popularity)",
    all_results['als_vs_baseline'],
    "Popularity",
    "ALS"
)

# Display results for SASRec vs Baseline
display_test_results(
    "SASRec vs Baseline (Popularity)",
    all_results['sasrec_vs_baseline'],
    "Popularity",
    "SASRec"
)

# Display results for ALS vs SASRec
display_test_results(
    "ALS vs SASRec",
    all_results['als_vs_sasrec'],
    "ALS",
    "SASRec"
)

st.markdown("--- ")
st.info("A significant p-value (typically < 0.05) suggests a statistically significant difference between the two recommenders for that metric.")

# --- New Section for Consumer Metrics (Churn and CLTV) ---
st.header("Overall Consumer Behavior Metrics (Simulated)")

if 'consumer_metrics' in all_results:
    consumer_metrics = all_results['consumer_metrics']

    st.subheader("Churn Metrics")
    churn_data = consumer_metrics.get('churn', {})
    st.write(f"**Overall Churn Rate:** {churn_data.get('overall_churn_rate', 0.0):.2%}")
    st.write(f"**Number of Churned Users (Simulated):** {churn_data.get('num_churned_users', 0)}")
    st.write(f"*(Based on a simulated {churn_data.get('total_users_considered', 0)} users)*")

    st.subheader("Customer Lifetime Value (CLTV) Metrics")
    cltv_data = consumer_metrics.get('cltv', {})
    st.write(f"**Overall Average CLTV:** ${cltv_data.get('overall_average_cltv', 0.0):.2f}")
    st.write(f"**CLTV Standard Deviation:** ${cltv_data.get('cltv_std_dev', 0.0):.2f}")
    st.write(f"*(Based on a simulated {cltv_data.get('total_users_considered', 0)} users)*")
else:
    st.warning("Consumer metrics (Churn, CLTV) not found in results. Please ensure the relevant cells have been run.")

st.markdown("--- ")
st.markdown("**Note on Consumer Metrics:** For a true assessment of recommender system impact on churn and CLTV, a long-running A/B test is required where different user groups are exposed to different recommendation strategies, and their subsequent long-term behavior (churn, purchases leading to CLTV) is tracked.")

Overwriting app.py
